In [1]:
# 초보자에게는 어려운 소스로 참고용으로 남겨놓습니다.

In [2]:
# 데이터 수집 : BeautifulSoup(스크래핑 : 1~2페이지)

In [3]:
# 로또 사이트 : 1. 몇회, 2. 당첨번호 6개, 3. 보너스 번호

In [4]:
# 페이지 HTML을 파싱하지 않고, 로또 사이트 내부 JSON API를 직접 호출한다.
api_url = "https://www.dhlottery.co.kr/lt645/selectPstLt645InfoNew.do"
latest_api_url = "https://www.dhlottery.co.kr/lt645/selectPstLt645Info.do"

In [5]:
import requests
import pandas as pd

In [6]:
headers = {
    "User-Agent": "Mozilla/5.0",
    "Referer": "https://www.dhlottery.co.kr/lt645/result",
    "Accept": "application/json, text/javascript, */*; q=0.01",
    "X-Requested-With": "XMLHttpRequest",
}

In [7]:
def parse_lotto_row(row):
    draw_date = row["ltRflYmd"]

    return {
        "회차": row["ltEpsd"],
        "추첨일": f"{draw_date[:4]}-{draw_date[4:6]}-{draw_date[6:]}",
        "번호1": row["tm1WnNo"],
        "번호2": row["tm2WnNo"],
        "번호3": row["tm3WnNo"],
        "번호4": row["tm4WnNo"],
        "번호5": row["tm5WnNo"],
        "번호6": row["tm6WnNo"],
        "보너스": row["bnsWnNo"],
        "1등당첨자수": row["rnk1WnNope"],
        "1등당첨금": row["rnk1WnAmt"],
        "총판매금액": row["wholEpsdSumNtslAmt"],
    }

In [8]:
def get_lotto(draw_no):
    params = {
        "srchDir": "center",
        "srchLtEpsd": str(draw_no),
    }
    response = requests.get(api_url, params=params, headers=headers, timeout=10)
    response.raise_for_status()

    data = response.json()
    rows = data.get("data", {}).get("list", [])
    row = next((item for item in rows if item.get("ltEpsd") == int(draw_no)), None)

    if row is None:
        raise ValueError(f"{draw_no}회 정보를 찾을 수 없습니다.")

    return parse_lotto_row(row)

In [9]:
def get_latest_lotto():
    response = requests.get(latest_api_url, headers=headers, timeout=10)
    response.raise_for_status()

    data = response.json()
    rows = data.get("data", {}).get("list", [])

    if not rows:
        raise ValueError("최신 회차 정보를 찾을 수 없습니다.")

    return parse_lotto_row(rows[0])

In [10]:
# 특정 회차 조회 예시
get_lotto(1103)

{'회차': 1103,
 '추첨일': '2024-01-20',
 '번호1': 10,
 '번호2': 12,
 '번호3': 29,
 '번호4': 31,
 '번호5': 40,
 '번호6': 44,
 '보너스': 2,
 '1등당첨자수': 17,
 '1등당첨금': 1574419633,
 '총판매금액': 110195070000}

In [11]:
# 최신 회차 조회 예시
get_latest_lotto()

{'회차': 1222,
 '추첨일': '2026-05-02',
 '번호1': 4,
 '번호2': 11,
 '번호3': 17,
 '번호4': 22,
 '번호5': 32,
 '번호6': 41,
 '보너스': 34,
 '1등당첨자수': 24,
 '1등당첨금': 1202986844,
 '총판매금액': 124243558000}

In [12]:
# 여러 회차를 표로 만들기
start_draw = 1100
end_draw = 1103

lotto_list = [get_lotto(draw_no) for draw_no in range(start_draw, end_draw + 1)]
df_lotto = pd.DataFrame(lotto_list)
df_lotto

,회차,추첨일,번호1,번호2,번호3,번호4,번호5,번호6,보너스,1등당첨자수,1등당첨금,총판매금액
0,1100,2023-12-30,17,26,29,30,31,43,12,13,2207575472,116187023000
1,1101,2024-01-06,6,7,13,28,36,42,41,13,2100529500,114758876000
2,1102,2024-01-13,13,14,22,26,37,38,20,20,1383591413,113178102000
3,1103,2024-01-20,10,12,29,31,40,44,2,17,1574419633,110195070000


In [13]:
# end